# Module 08-3 솔루션: Retry + Circuit Breaker 통합 에이전트

In [ ]:
import sys
sys.path.insert(0, '../..')

import time
import threading
from enum import Enum
from dataclasses import dataclass, field
from functools import wraps
from typing import TypedDict

from langgraph.graph import StateGraph, END
from common.fake_llm import FakeLLM

print("환경 설정 완료")

In [ ]:
# CircuitBreaker (재사용)
class CircuitState(Enum):
    CLOSED = "closed"
    OPEN = "open"
    HALF_OPEN = "half_open"

class CircuitOpenError(Exception):
    def __init__(self, name, remaining_seconds):
        self.name = name
        self.remaining_seconds = remaining_seconds
        super().__init__(f"Circuit '{name}'이 OPEN 상태. {remaining_seconds:.1f}초 후 재시도.")

@dataclass
class CircuitBreaker:
    name: str
    failure_threshold: int = 5
    recovery_timeout: float = 60.0
    expected_exceptions: tuple = (Exception,)
    _state: CircuitState = field(default=CircuitState.CLOSED, init=False)
    _failure_count: int = field(default=0, init=False)
    _last_failure_time: float = field(default=0.0, init=False)
    _lock: threading.Lock = field(default_factory=threading.Lock, init=False)

    @property
    def state(self):
        with self._lock:
            if self._state == CircuitState.OPEN:
                if time.monotonic() - self._last_failure_time >= self.recovery_timeout:
                    self._state = CircuitState.HALF_OPEN
            return self._state

    def _record_success(self):
        with self._lock:
            self._failure_count = 0
            if self._state == CircuitState.HALF_OPEN:
                self._state = CircuitState.CLOSED

    def _record_failure(self):
        with self._lock:
            self._failure_count += 1
            self._last_failure_time = time.monotonic()
            if self._failure_count >= self.failure_threshold:
                self._state = CircuitState.OPEN

    def call(self, fn, *args, **kwargs):
        if self.state == CircuitState.OPEN:
            remaining = self.recovery_timeout - (time.monotonic() - self._last_failure_time)
            raise CircuitOpenError(self.name, max(0, remaining))
        try:
            result = fn(*args, **kwargs)
            self._record_success()
            return result
        except self.expected_exceptions:
            self._record_failure()
            raise

print("CircuitBreaker 로드 완료")

In [ ]:
# 실습 1 솔루션: @retry_on_llm_error 데코레이터
RETRIABLE_EXCEPTIONS = (ConnectionError, TimeoutError)

def retry_on_llm_error(max_retries=3, base_delay=1.0, max_delay=30.0, backoff_factor=2.0):
    """LLM 호출 노드용 재시도 데코레이터."""
    def decorator(node_fn):
        @wraps(node_fn)
        def wrapper(state, **kwargs):
            last_exception = None

            for attempt in range(max_retries + 1):
                try:
                    return node_fn(state, **kwargs)

                except RETRIABLE_EXCEPTIONS as exc:
                    last_exception = exc
                    if attempt < max_retries:
                        delay = min(base_delay * (backoff_factor ** attempt), max_delay)
                        print(f"재시도 {attempt + 1}/{max_retries}: {delay:.2f}초 대기")
                        time.sleep(delay)

                except Exception as exc:
                    # Non-retriable: 즉시 에러 반환
                    return {
                        "error": f"[{node_fn.__name__}] {type(exc).__name__}: {exc}",
                        "current_step": node_fn.__name__,
                    }

            # 모든 재시도 실패
            return {
                "error": f"[{node_fn.__name__}] 모든 재시도 실패: {last_exception}",
                "current_step": node_fn.__name__,
            }

        return wrapper
    return decorator


# 검증
class TestState(TypedDict):
    query: str
    result: str | None
    error: str | None
    current_step: str

@retry_on_llm_error(max_retries=2, base_delay=0.01)
def node_non_retriable(state):
    raise ValueError("잘못된 입력")

result = node_non_retriable({"query": "test", "result": None, "error": None, "current_step": "start"})
assert "error" in result and result["error"] is not None
print(f"Non-retriable 에러 처리 통과")

retry_count = 0
@retry_on_llm_error(max_retries=3, base_delay=0.01)
def node_retriable(state):
    global retry_count
    retry_count += 1
    if retry_count < 3:
        raise ConnectionError("일시적 오류")
    return {"result": "성공", "current_step": "done"}

retry_count = 0
result = node_retriable({"query": "test", "result": None, "error": None, "current_step": "start"})
assert result.get("result") == "성공"
print(f"Retriable 에러 재시도 통과: {retry_count}번 시도")
print("실습 1 완료!")

In [ ]:
# 실습 2 솔루션: 3노드 에이전트

class TaskState(TypedDict):
    query: str
    fetched_data: str | None
    analysis: str | None
    ticket_key: str | None
    error: str | None
    current_step: str


api_circuit = CircuitBreaker("api", failure_threshold=5, expected_exceptions=(ConnectionError,))
ticket_circuit = CircuitBreaker("ticket", failure_threshold=5, expected_exceptions=(ConnectionError,))
llm = FakeLLM(responses={"분석": "버그 발견: 인증 로직 누락", "코드": "코드 분석 완료"})


def fetch_data_node(state: TaskState) -> dict:
    try:
        data = api_circuit.call(
            lambda: f"데이터 수집 완료: {state['query']}"
        )
        return {"fetched_data": data, "current_step": "fetch_data"}
    except CircuitOpenError as exc:
        return {"error": f"API 서킷 열림: {exc}", "current_step": "fetch_data"}
    except Exception as exc:
        return {"error": f"데이터 가져오기 실패: {exc}", "current_step": "fetch_data"}


@retry_on_llm_error(max_retries=2, base_delay=0.01)
def analyze_node(state: TaskState) -> dict:
    if not state.get("fetched_data"):
        raise ValueError("분석할 데이터 없음")
    result = llm.invoke(f"분석: {state['fetched_data']}").content
    return {"analysis": result, "current_step": "analyze"}


def create_ticket_node(state: TaskState) -> dict:
    try:
        key = ticket_circuit.call(lambda: "TICKET-001")
        return {"ticket_key": key, "current_step": "create_ticket"}
    except CircuitOpenError as exc:
        return {"error": f"티켓 서킷 열림: {exc}", "current_step": "create_ticket"}
    except Exception as exc:
        return {"error": f"티켓 생성 실패: {exc}", "current_step": "create_ticket"}


def handle_error_node(state: TaskState) -> dict:
    print(f"[에러 처리] {state.get('error')}")
    return {"current_step": "handle_error"}


def route_after_fetch(state: TaskState) -> str:
    return "handle_error" if state.get("error") else "analyze"


def route_after_analyze(state: TaskState) -> str:
    return "handle_error" if state.get("error") else "create_ticket"


# 그래프 구성
graph = StateGraph(TaskState)
graph.add_node("fetch_data", fetch_data_node)
graph.add_node("analyze", analyze_node)
graph.add_node("create_ticket", create_ticket_node)
graph.add_node("handle_error", handle_error_node)
graph.set_entry_point("fetch_data")
graph.add_conditional_edges("fetch_data", route_after_fetch,
    {"analyze": "analyze", "handle_error": "handle_error"})
graph.add_conditional_edges("analyze", route_after_analyze,
    {"create_ticket": "create_ticket", "handle_error": "handle_error"})
graph.add_edge("create_ticket", END)
graph.add_edge("handle_error", END)

app = graph.compile()

result = app.invoke({
    "query": "로그인 API 코드를 분석해주세요",
    "fetched_data": None, "analysis": None,
    "ticket_key": None, "error": None,
    "current_step": "start",
})

assert result.get("error") is None
assert result.get("fetched_data") is not None
print(f"정상 케이스 통과! 티켓: {result.get('ticket_key')}, 분석: {result.get('analysis')}")
print("실습 2 완료! 모든 실습 완료")